# Chcek SHAP arrays for all datasets and fix if needed


## 1. Environment Setup & Global Configuration
Definition of base paths (`BASE_PATH = "shared/explain-ts/ds"`), tracking directories (e.g., timestamped run logs for March 1, 2026), and strict typing imports.


In [24]:
import gc
import os
import pickle
# conda install -c conda-forge shap
# conda install -c conda-forge ipywidgets
# pip install "tensorflow[and-cuda]"
import shutil
import zipfile
import sys
import time
import warnings
from typing import Any
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import shap
import tensorflow as tf
from scipy.stats import percentileofscore
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.initializers import GlorotUniform, Orthogonal, Zeros
from tensorflow.keras.layers import ConvLSTM1D, Input, Reshape, Dropout, Flatten, Dense
from tensorflow.keras.models import Sequential, load_model

print(tf.config.list_physical_devices('GPU'))

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # 0 - first gpu, 1 - second, "0,1" - both gpu, first used, "-1" - none

os.environ['LD_LIBRARY_PATH'] = f"{sys.prefix}/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"

# The target directory structure expected by the rest of the notebook
BASE_PATH = "shared/explain-ts/ds"
# BASE_PATH = "shared/UCI-Benchmark/ds"

UNI_DIR = os.path.join(BASE_PATH, "univariate")
MULTI_DIR = os.path.join(BASE_PATH, "multivariate")


# Load model

class SafeConvLSTM1D(ConvLSTM1D):
    def __init__(self, *args, **kwargs):
        kwargs.pop('time_major', None)
        super().__init__(*args, **kwargs)


class SafeGlorotUniform(tf.keras.initializers.GlorotUniform):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)
        super().__init__(**kwargs)


class SafeOrthogonal(tf.keras.initializers.Orthogonal):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)
        super().__init__(**kwargs)


class SafeZeros(tf.keras.initializers.Zeros):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)  # Zeros might occasionally throw it too
        super().__init__()


# Crucial step: map the standard Keras names to our Safe wrappers
CUSTOM_OBJECTS = {
    'GlorotUniform': SafeGlorotUniform,
    'Orthogonal': SafeOrthogonal,
    'Zeros': SafeZeros,
    'ConvLSTM1D': SafeConvLSTM1D,
    'SafeConvLSTM1D': SafeConvLSTM1D
}


# --- Robust Loader ---
def load_benchmark_model(dataset_path: str, input_shape: tuple, num_classes: int) -> tf.keras.Model:
    h5_path = os.path.join(dataset_path, 'model.h5')
    tf_dir = os.path.join(dataset_path, 'model_tf/1')

    # 1. Standard load if healthy H5 exists
    if os.path.exists(h5_path):
        # We MUST pass CUSTOM_OBJECTS here to intercept 'dtype' during from_config()
        return load_model(h5_path, custom_objects=CUSTOM_OBJECTS, compile=False)

    # 2. Repair & Repack via Checkpoint Injection
    if os.path.isdir(tf_dir):
        print(f"Repacking legacy model for {os.path.basename(dataset_path)}...")

        # Build identical architecture using Safe layers to avoid initialization errors
        model = Sequential([
            Input(shape=input_shape),
            Reshape((*input_shape, 1), name='reshape'),
            SafeConvLSTM1D(64, kernel_size=3, padding='same', return_sequences=True, name='conv_lstm1d'),
            SafeConvLSTM1D(32, kernel_size=3, padding='same', return_sequences=True, name='conv_lstm1d_1'),
            Dropout(0.2, name='dropout'),
            Flatten(name='embedding'),
            Dense(100, activation='relu', name='dense'),
            Dense(num_classes, activation='softmax', name='dense_1')
        ])

        ckpt_prefix = os.path.join(tf_dir, 'variables', 'variables')

        try:
            checkpoint = tf.train.Checkpoint(model=model)
            checkpoint.restore(ckpt_prefix).expect_partial()
        except Exception as e:
            print(f"Checkpoint restore warning: {e}. Trying native Keras load_weights...")
            model.load_weights(ckpt_prefix)

        # Save healthy version for future runs
        model.save(h5_path)
        print("Successfully repacked to clean model.h5!")
        return model

    raise FileNotFoundError(f"No model artifacts found in {dataset_path}")


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 2. Dataset Auditing & Explainer Availability
A fast, lightweight pass over the dataset registry to identify which datasets possess the required SHAP or LIME artifacts. Only fully validated datasets are queued for the main extraction loop.


In [2]:
def audit_datasets(categories_paths: Dict[str, str]) -> List[str]:
    """
    Iterates over all datasets to ensure they contain the required test data,
    a loadable Keras model, and at least one continuous explainer (SHAP or LIME).
    Fails fast if critical artifacts or both explainers are missing.
    Clears Keras session continuously to prevent OOM errors.

    Returns:
        List of absolute paths to fully verified datasets ready for PHAR extraction.
    """
    verified_datasets = []

    print("Starting Dataset Auditing & Explainer Availability Check...\n")

    for category, cat_path in categories_paths.items():
        if not os.path.exists(cat_path):
            print(f"Skipping {category}: Directory not found at {cat_path}")
            continue

        for ds_name in sorted(os.listdir(cat_path)):
            ds_path = os.path.join(cat_path, ds_name)
            if not os.path.isdir(ds_path):
                continue

            # 1. Check core data existence
            train_x_path = os.path.join(ds_path, 'trainX.pickle')
            train_y_path = os.path.join(ds_path, 'trainy.pickle')
            test_x_path = os.path.join(ds_path, 'testX.pickle')
            test_y_path = os.path.join(ds_path, 'testy.pickle')

            assert os.path.exists(train_x_path), f"FAIL FAST: Missing trainX.pickle in {ds_name}"
            assert os.path.exists(train_y_path), f"FAIL FAST: Missing trainy.pickle in {ds_name}"
            assert os.path.exists(test_x_path), f"FAIL FAST: Missing testX.pickle in {ds_name}"
            assert os.path.exists(test_y_path), f"FAIL FAST: Missing testy.pickle in {ds_name}"

            # 2. Check explainer existence
            shap_path = os.path.join(ds_path, 'svts.pickle')
            lime_path = os.path.join(ds_path, 'lvts.pickle')

            has_shap = os.path.exists(shap_path)
            has_lime = os.path.exists(lime_path)

            if not has_shap and not has_lime:
                raise FileNotFoundError(f"FAIL FAST: No SHAP or LIME artifacts found for {ds_name}!")
            elif not has_shap or not has_lime:
                missing = "SHAP" if not has_shap else "LIME"
                print(f"WARN: [{ds_name}] is missing {missing} explanations. Proceeding with available explainer.")

            # 3. Verify data loading & dimensions
            with open(test_x_path, 'rb') as f:
                testX = pickle.load(f)
            with open(test_y_path, 'rb') as f:
                testy = pickle.load(f)

            input_dim = testX.shape[1:]
            num_classes = testy.shape[1] if len(testy.shape) > 1 else len(np.unique(testy))

            # 4. Verify model loading
            try:
                model = load_benchmark_model(ds_path, input_shape=input_dim, num_classes=num_classes)
            except Exception as e:
                raise RuntimeError(f"FAIL FAST: Could not load model for {ds_name}. Error: {e}")

            # 5. Strict memory cleanup to prevent OOM in loop
            del model
            del testX
            del testy
            tf.keras.backend.clear_session()
            gc.collect()

            verified_datasets.append(ds_path)

    print(f"\nAudit complete. Successfully verified {len(verified_datasets)} datasets.")
    return verified_datasets


In [3]:
categories_to_audit = {
    "univariate": UNI_DIR,
    "multivariate": MULTI_DIR
}

verified_dataset_paths = audit_datasets(categories_to_audit)

Starting Dataset Auditing & Explainer Availability Check...



/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1772396795.888449    4966 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1038 MB memory:  -> device: 0, name: NVIDIA RTX A5500, pci bus id: 0000:51:00.0, compute capability: 8.6
I0000 00:00:1772396795.888949    4966 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22198 MB memory:  -> device: 1, name: NVIDIA RTX A5500, pci bus id: 0000:9c:00.0, compute capability: 8.6
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input

WARN: [FaceDetection] is missing LIME explanations. Proceeding with available explainer.

Audit complete. Successfully verified 103 datasets.


In [3]:
verified_dataset_paths = ['shared/explain-ts/ds/univariate/Adiac',
                          'shared/explain-ts/ds/univariate/BME',
                          'shared/explain-ts/ds/univariate/Beef',
                          'shared/explain-ts/ds/univariate/BeetleFly',
                          'shared/explain-ts/ds/univariate/BirdChicken',
                          'shared/explain-ts/ds/univariate/CBF',
                          'shared/explain-ts/ds/univariate/Chinatown',
                          'shared/explain-ts/ds/univariate/Coffee',
                          'shared/explain-ts/ds/univariate/Computers',
                          'shared/explain-ts/ds/univariate/CricketX',
                          'shared/explain-ts/ds/univariate/CricketY',
                          'shared/explain-ts/ds/univariate/CricketZ',
                          'shared/explain-ts/ds/univariate/Crop',
                          'shared/explain-ts/ds/univariate/DiatomSizeReduction',
                          'shared/explain-ts/ds/univariate/DistalPhalanxOutlineAgeGroup',
                          'shared/explain-ts/ds/univariate/DistalPhalanxOutlineCorrect',
                          'shared/explain-ts/ds/univariate/DistalPhalanxTW',
                          'shared/explain-ts/ds/univariate/DodgerLoopDay',
                          'shared/explain-ts/ds/univariate/DodgerLoopGame',
                          'shared/explain-ts/ds/univariate/DodgerLoopWeekend',
                          'shared/explain-ts/ds/univariate/ECG200',
                          'shared/explain-ts/ds/univariate/ECG5000',
                          'shared/explain-ts/ds/univariate/ECGFiveDays',
                          'shared/explain-ts/ds/univariate/Earthquakes',
                          'shared/explain-ts/ds/univariate/ElectricDevices',
                          'shared/explain-ts/ds/univariate/FaceFour',
                          'shared/explain-ts/ds/univariate/FiftyWords',
                          'shared/explain-ts/ds/univariate/FordA',
                          'shared/explain-ts/ds/univariate/FordB',
                          'shared/explain-ts/ds/univariate/FreezerRegularTrain',
                          'shared/explain-ts/ds/univariate/FreezerSmallTrain',
                          'shared/explain-ts/ds/univariate/Fungi',
                          'shared/explain-ts/ds/univariate/GunPoint',
                          'shared/explain-ts/ds/univariate/GunPointAgeSpan',
                          'shared/explain-ts/ds/univariate/GunPointMaleVersusFemale',
                          'shared/explain-ts/ds/univariate/GunPointOldVersusYoung',
                          'shared/explain-ts/ds/univariate/Herring',
                          'shared/explain-ts/ds/univariate/InsectWingbeatSound',
                          'shared/explain-ts/ds/univariate/ItalyPowerDemand',
                          'shared/explain-ts/ds/univariate/LargeKitchenAppliances',
                          'shared/explain-ts/ds/univariate/Lightning2',
                          'shared/explain-ts/ds/univariate/Lightning7',
                          'shared/explain-ts/ds/univariate/Meat',
                          'shared/explain-ts/ds/univariate/MedicalImages',
                          'shared/explain-ts/ds/univariate/MiddlePhalanxOutlineAgeGroup',
                          'shared/explain-ts/ds/univariate/MiddlePhalanxOutlineCorrect',
                          'shared/explain-ts/ds/univariate/MiddlePhalanxTW',
                          'shared/explain-ts/ds/univariate/MoteStrain',
                          'shared/explain-ts/ds/univariate/OSULeaf',
                          'shared/explain-ts/ds/univariate/OliveOil',
                          'shared/explain-ts/ds/univariate/PhalangesOutlinesCorrect',
                          'shared/explain-ts/ds/univariate/Plane',
                          'shared/explain-ts/ds/univariate/PowerCons',
                          'shared/explain-ts/ds/univariate/ProximalPhalanxOutlineAgeGroup',
                          'shared/explain-ts/ds/univariate/ProximalPhalanxOutlineCorrect',
                          'shared/explain-ts/ds/univariate/ProximalPhalanxTW',
                          'shared/explain-ts/ds/univariate/RefrigerationDevices',
                          'shared/explain-ts/ds/univariate/ScreenType',
                          'shared/explain-ts/ds/univariate/ShapeletSim',
                          'shared/explain-ts/ds/univariate/ShapesAll',
                          'shared/explain-ts/ds/univariate/SmallKitchenAppliances',
                          'shared/explain-ts/ds/univariate/SmoothSubspace',
                          'shared/explain-ts/ds/univariate/SonyAIBORobotSurface1',
                          'shared/explain-ts/ds/univariate/SonyAIBORobotSurface2',
                          'shared/explain-ts/ds/univariate/Strawberry',
                          'shared/explain-ts/ds/univariate/SwedishLeaf',
                          'shared/explain-ts/ds/univariate/Symbols',
                          'shared/explain-ts/ds/univariate/SyntheticControl',
                          'shared/explain-ts/ds/univariate/ToeSegmentation2',
                          'shared/explain-ts/ds/univariate/Trace',
                          'shared/explain-ts/ds/univariate/TwoLeadECG',
                          'shared/explain-ts/ds/univariate/TwoPatterns',
                          'shared/explain-ts/ds/univariate/UMD',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryAll',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryX',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryY',
                          'shared/explain-ts/ds/univariate/UWaveGestureLibraryZ',
                          'shared/explain-ts/ds/univariate/Wafer',
                          'shared/explain-ts/ds/univariate/Wine',
                          'shared/explain-ts/ds/univariate/WordSynonyms',
                          'shared/explain-ts/ds/univariate/Worms',
                          'shared/explain-ts/ds/univariate/WormsTwoClass',
                          'shared/explain-ts/ds/univariate/Yoga',
                          'shared/explain-ts/ds/multivariate/ArticularyWordRecognition',
                          'shared/explain-ts/ds/multivariate/AtrialFibrillation',
                          'shared/explain-ts/ds/multivariate/BasicMotions',
                          'shared/explain-ts/ds/multivariate/Cricket',
                          'shared/explain-ts/ds/multivariate/ERing',
                          'shared/explain-ts/ds/multivariate/Epilepsy',
                          'shared/explain-ts/ds/multivariate/EthanolConcentration',
                          'shared/explain-ts/ds/multivariate/FaceDetection',
                          'shared/explain-ts/ds/multivariate/FingerMovements',
                          'shared/explain-ts/ds/multivariate/HandMovementDirection',
                          'shared/explain-ts/ds/multivariate/Handwriting',
                          'shared/explain-ts/ds/multivariate/Heartbeat',
                          'shared/explain-ts/ds/multivariate/LSST',
                          'shared/explain-ts/ds/multivariate/Libras',
                          'shared/explain-ts/ds/multivariate/NATOPS',
                          'shared/explain-ts/ds/multivariate/PenDigits',
                          'shared/explain-ts/ds/multivariate/RacketSports',
                          'shared/explain-ts/ds/multivariate/SelfRegulationSCP1',
                          'shared/explain-ts/ds/multivariate/SelfRegulationSCP2',
                          'shared/explain-ts/ds/multivariate/UWaveGestureLibrary']

## 3. Core Classes: 3D-Aware Rule Generator
Implementation of the `GroundTruthRuleGenerator` adapted natively for 3D time-series formats `(n_samples, n_timesteps, n_variables)`. This includes overriding the perturbation mechanisms to handle temporal dimensions and abstracting the prediction logic for Keras `ConvLSTM-based` architectures.


In [4]:
def format_explanations_to_4d(explanations: Any, X_shape: tuple, num_classes: int) -> Tuple[np.ndarray, bool]:
    """
    Helper to ensure explanations are strictly shaped as (N, C, T, V).
    ExplainTS SHAP might be stored as a list of arrays or (N, T, V).

    Returns:
        A tuple (formatted_array, success_flag).
        success_flag is False if the array consists entirely of NaNs.
    """
    N, T, V = X_shape
    formatted_array = None

    if isinstance(explanations, list) and len(explanations) == num_classes:
        # e.g. List of C arrays, each (N, T, V)
        formatted_array = np.stack(explanations, axis=1)
    elif isinstance(explanations, np.ndarray):
        if explanations.ndim == 3:  # (N, T, V) for binary
            # Duplicate across classes for demonstration if missing class dim
            formatted_array = np.stack([explanations] * num_classes, axis=1)
        elif explanations.ndim == 4:
            formatted_array = explanations

    if formatted_array is None:
        raise ValueError(f"Unrecognized explanation shape/type: {type(explanations)}")

    # Check if the entire array consists of NaNs
    if np.isnan(formatted_array).all():
        print("WARN: Formatted explanation array contains ONLY NaN values.")
        return formatted_array, False

    return formatted_array, True


def get_stratified_pool(
        indices: np.ndarray,
        X: np.ndarray,
        expl: np.ndarray,
        y: np.ndarray,
        pool_fraction: float = 0.1,
        random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Safely extracts a stratified subset from the data based on a fraction.
    Bypasses sklearn's limitation with singleton classes and guarantees
    mathematical bounds for sample size.
    """
    total_samples = len(y)
    unique_classes, counts = np.unique(y, return_counts=True)
    num_classes = len(unique_classes)

    # Calculate target pool size based on fraction
    calculated_size = int(total_samples * pool_fraction)

    # Guard 1: Ensure enough samples to represent at least one of each class
    pool_size = max(calculated_size, num_classes)

    # Guard 2: Cap at the maximum available samples
    pool_size = min(pool_size, total_samples)

    # 1. Isolate singletons
    singleton_classes = unique_classes[counts == 1]
    singleton_mask = np.isin(y, singleton_classes)
    multiple_mask = ~singleton_mask

    indices_single = indices[singleton_mask]
    X_single = X[singleton_mask]
    expl_single = expl[singleton_mask]
    y_single = y[singleton_mask]

    remaining_size = pool_size - len(indices_single)

    # 2. Sample the rest of the data
    if remaining_size > 0 and multiple_mask.sum() > 0:
        # Guard 3: Do not request more samples than available in the non-singleton subset
        remaining_size = min(remaining_size, int(multiple_mask.sum()))

        try:
            indices_rest, _, X_rest, _, expl_rest, _, y_rest, _ = train_test_split(
                indices[multiple_mask],
                X[multiple_mask],
                expl[multiple_mask],
                y[multiple_mask],
                train_size=remaining_size,
                random_state=random_state,
                stratify=y[multiple_mask]
            )
        except ValueError as e:
            print(f"WARN: Stratification failed ({e}). Falling back to unstratified split.")
            indices_rest, _, X_rest, _, expl_rest, _, y_rest, _ = train_test_split(
                indices[multiple_mask],
                X[multiple_mask],
                expl[multiple_mask],
                y[multiple_mask],
                train_size=remaining_size,
                random_state=random_state,
                stratify=None
            )

        indices_pool = np.concatenate([indices_single, indices_rest])
        X_pool = np.concatenate([X_single, X_rest])
        expl_pool = np.concatenate([expl_single, expl_rest])
        y_pool = np.concatenate([y_single, y_rest])
    else:
        # If singletons exceed or match the requested pool size, just slice them
        indices_pool = indices_single[:pool_size]
        X_pool = X_single[:pool_size]
        expl_pool = expl_single[:pool_size]
        y_pool = y_single[:pool_size]

    return indices_pool, X_pool, expl_pool, y_pool




In [8]:
class PHARRuleGenerator(BaseEstimator, TransformerMixin):
    def __init__(self,
                 model: Any,
                 threshold_percentile: float = 40.0,
                 use_global_importance: bool = False,
                 perturb_sigma: float = 1.0,
                 perturbation_samples_count: int = 10_000,
                 min_selected_features: int = 1,
                 topk_fallback: int = 0):

        self.model = model
        self.threshold_percentile = float(threshold_percentile)
        self.use_global_importance = use_global_importance
        self.perturb_sigma = perturb_sigma
        self.perturbation_samples_count = perturbation_samples_count
        self.min_selected_features = int(min_selected_features)
        self.topk_fallback = int(topk_fallback)

        self.n_timesteps = 0
        self.n_variables = 0
        self.n_classes = 0
        self.feature_names = []
        self.feature_coords = []

        self.class_thresholds = {}
        self.class_all_abs_explanations = {}
        self.class_abs_explanations_per_feature = {}
        self.feature_stats = {}

    def fit(self, X_train: np.ndarray, expl_train: np.ndarray) -> "PHARRuleGenerator":
        assert X_train.ndim == 3, f"X_train must be 3D (N, T, V), got {X_train.ndim}D"
        assert expl_train.ndim == 4, f"expl_train must be 4D (N, C, T, V), got {expl_train.ndim}D"

        n_samples, self.n_timesteps, self.n_variables = X_train.shape
        self.n_classes = expl_train.shape[1]

        for t in range(self.n_timesteps):
            for v in range(self.n_variables):
                if self.n_variables == 1:
                    self.feature_names.append(f"feature_{t}")
                else:
                    self.feature_names.append(f"var_{v}_ts_{t}")
                self.feature_coords.append((t, v))

        for class_idx in range(self.n_classes):
            sliced_expl = expl_train[:, class_idx, :, :]
            sliced_flat = sliced_expl.reshape(n_samples, -1)

            self.class_thresholds[class_idx] = {
                f_name: np.percentile(np.abs(sliced_flat[:, i]), self.threshold_percentile)
                for i, f_name in enumerate(self.feature_names)
            }

            self.class_all_abs_explanations[class_idx] = np.abs(sliced_flat).ravel()
            self.class_abs_explanations_per_feature[class_idx] = {
                f_name: np.abs(sliced_flat[:, i])
                for i, f_name in enumerate(self.feature_names)
            }

        X_flat = X_train.reshape(n_samples, -1)
        self.feature_stats = {
            f_name: {
                "mean": X_flat[:, i].mean(),
                "std": X_flat[:, i].std(),
                "min": X_flat[:, i].min(),
                "max": X_flat[:, i].max()
            }
            for i, f_name in enumerate(self.feature_names)
        }
        return self

    def transform(self, X_test: np.ndarray, expl_test: np.ndarray, original_indices: Optional[List[int]] = None) -> \
            List[Dict]:
        y_pred_proba = self.model.predict(X_test, verbose=0)
        y_pred_classes = np.argmax(y_pred_proba, axis=1)

        if original_indices is None:
            original_indices = list(range(X_test.shape[0]))

        rules = []

        for idx in range(X_test.shape[0]):
            start_time = time.time()
            instance = X_test[idx:idx + 1]
            original_prediction = y_pred_classes[idx]
            real_index = original_indices[idx]

            weights_for_pred = expl_test[idx, original_prediction, :, :].ravel()
            selected_features = []

            for i, f_name in enumerate(self.feature_names):
                abs_weight = abs(weights_for_pred[i])
                if self.use_global_importance:
                    exp_global_percentile = percentileofscore(self.class_all_abs_explanations[original_prediction],
                                                              abs_weight)
                    exceeds = exp_global_percentile >= self.threshold_percentile
                else:
                    exceeds = abs_weight >= self.class_thresholds[original_prediction][f_name]

                if exceeds:
                    f_stats = self.feature_stats[f_name]
                    selected_features.append((i, f_name, f_stats["min"], f_stats["max"], f_stats["std"]))

            if len(selected_features) < self.min_selected_features:
                print(f"WARN: Rule {idx} has less than {self.min_selected_features} selected features. ")
                if self.topk_fallback > 0:
                    top_idx = np.argsort(np.abs(weights_for_pred))[::-1]
                    top_idx = np.argsort(np.abs(weights_for_pred))[::-1]
                    used = {f_name for (_, f_name, *_) in selected_features}
                    added = 0
                    need = max(self.topk_fallback, self.min_selected_features - len(selected_features))

                    for fi in top_idx:
                        f_name = self.feature_names[fi]
                        if f_name not in used:
                            f_stats = self.feature_stats[f_name]
                            selected_features.append((fi, f_name, f_stats["min"], f_stats["max"], f_stats["std"]))
                            used.add(f_name)
                            added += 1
                            if added >= need:
                                break

            rule = {}
            confidence = 0.0
            coverage = 0.0

            if selected_features:

                perturbed_samples = []
                perturbed_metadata = []

                for _ in range(self.perturbation_samples_count):
                    p_sample = instance.copy()
                    meta_for_this_sample = []

                    for (fi, f_name, f_min, f_max, f_std) in selected_features:
                        t, v = self.feature_coords[fi]
                        orig_val = instance[0, t, v]
                        random_val = np.random.uniform(orig_val - self.perturb_sigma * f_std,
                                                       orig_val + self.perturb_sigma * f_std)
                        p_sample[0, t, v] = random_val
                        meta_for_this_sample.append((f_name, random_val))

                    perturbed_samples.append(p_sample[0])
                    perturbed_metadata.append(meta_for_this_sample)

                perturbed_array = np.array(perturbed_samples)
                p_preds = np.argmax(self.model.predict(perturbed_array, verbose=0), axis=1)

                pred_consistent_values = {}

                for sample_metadata, p_class in zip(perturbed_metadata, p_preds):
                    if p_class == original_prediction:
                        for f_name, val in sample_metadata:
                            pred_consistent_values.setdefault(f_name, []).append(val)

                for f_name, values in pred_consistent_values.items():
                    if len(values) == 1:
                        print(f"WARN: Rule {idx} has only 1 consistent value for feature {f_name}.")
                        val = values[0]
                        f_stats = self.feature_stats[f_name]
                        values.extend([
                            max(val - f_stats["std"], f_stats["min"]),
                            min(val + f_stats["std"], f_stats["max"])
                        ])

                    if len(values) > 1:
                        f_min, f_max = min(values), max(values)
                        rule[f_name] = [f">{f_min}", f"<={f_max}"]

                coverage, confidence = self._compute_coverage_and_confidence(rule, X_test, y_pred_classes,
                                                                             original_prediction)

            inference_time = time.time() - start_time

            rules.append({
                "index": int(real_index),
                "success": bool(rule),
                "prediction": int(original_prediction),
                "rule": rule,
                "confidence": confidence,
                "coverage": coverage,
                "exp_count": len(rule.keys()),
                "time_inference": inference_time,
                "method": "PHAR",
                "threshold_percentile": self.threshold_percentile,
                "use_global_importance": self.use_global_importance,
                "perturb_sigma": self.perturb_sigma,
                "perturbation_samples_count": self.perturbation_samples_count
            })

        return rules

    def _compute_coverage_and_confidence(self, rule: Dict[str, List[str]], X: np.ndarray,
                                         y_pred: np.ndarray, reference_class: int) -> Tuple[float, float]:
        if not rule:
            return 0.0, 0.0

        mask = np.ones(X.shape[0], dtype=bool)

        for f_name, interval in rule.items():
            lower_val = float(interval[0][1:])
            upper_val = float(interval[1][2:])

            fi = self.feature_names.index(f_name)
            t, v = self.feature_coords[fi]

            current_mask = (X[:, t, v] > lower_val) & (X[:, t, v] <= upper_val)
            mask = mask & current_mask

        coverage_value = mask.mean()
        if coverage_value == 0:
            return 0.0, 0.0

        covered_indices = np.where(mask)[0]
        confidence_value = np.mean(y_pred[covered_indices] == reference_class)
        return float(coverage_value), float(confidence_value)


def format_explanations_to_4d_strict(explanations: Any, expected_samples: int, num_classes: int, T: int,
                                     V: int) -> np.ndarray:
    """
    Helper to ensure explanations are strictly shaped as (N, C, T, V).
    Used internally by the fallback mechanism to standardize SHAP outputs.
    """
    if isinstance(explanations, list):
        if len(explanations) == num_classes:
            formatted_array = np.stack(explanations, axis=1)
        elif len(explanations) == 1 and num_classes == 2:
            # Binary classification edge case in some SHAP versions
            base_arr = explanations[0]
            formatted_array = np.stack([-base_arr, base_arr], axis=1)
        else:
            raise ValueError(f"Unexpected SHAP list length: {len(explanations)} for {num_classes} classes.")
    elif isinstance(explanations, np.ndarray):
        if explanations.ndim == 3:
            formatted_array = np.stack([explanations] * num_classes, axis=1)
        elif explanations.ndim == 4:
            formatted_array = explanations
        else:
            raise ValueError(f"Unexpected SHAP array ndim: {explanations.ndim}")
    else:
        raise ValueError(f"Unrecognized SHAP output type: {type(explanations)}")

    assert formatted_array.shape == (expected_samples, num_classes, T, V), \
        f"Shape mismatch. Expected {(expected_samples, num_classes, T, V)}, got {formatted_array.shape}"

    return formatted_array


def format_explanations_to_4d_strict(explanations: Any, expected_samples: int, num_classes: int, T: int,
                                     V: int) -> np.ndarray:
    """
    Helper to ensure explanations are strictly shaped as (N, C, T, V).
    Used internally by the fallback mechanism to standardize SHAP outputs.
    Automatically handles SHAP returning (N, T, V, C) by transposing the axes.
    """
    if isinstance(explanations, list):
        if len(explanations) == num_classes:
            formatted_array = np.stack(explanations, axis=1)
        elif len(explanations) == 1 and num_classes == 2:
            base_arr = explanations[0]
            formatted_array = np.stack([-base_arr, base_arr], axis=1)
        else:
            raise ValueError(f"Unexpected SHAP list length: {len(explanations)} for {num_classes} classes.")

    elif isinstance(explanations, np.ndarray):
        if explanations.ndim == 3:
            formatted_array = np.stack([explanations] * num_classes, axis=1)
        elif explanations.ndim == 4:
            # Check if SHAP returned (N, T, V, C) instead of (N, C, T, V)
            if explanations.shape == (expected_samples, T, V, num_classes):
                # Transpose from (0, 1, 2, 3) -> (0, 3, 1, 2)
                formatted_array = np.transpose(explanations, (0, 3, 1, 2))
            else:
                formatted_array = explanations
        else:
            raise ValueError(f"Unexpected SHAP array ndim: {explanations.ndim}")
    else:
        raise ValueError(f"Unrecognized SHAP output type: {type(explanations)}")

    assert formatted_array.shape == (expected_samples, num_classes, T, V), \
        f"Shape mismatch. Expected {(expected_samples, num_classes, T, V)}, got {formatted_array.shape}"

    return formatted_array


def compute_shap_in_batches(explainer: shap.GradientExplainer, X: np.ndarray, batch_size: int = 128,
                            cache_dir: str = None) -> Any:
    """
    Computes SHAP values in chunks to prevent OOM errors on GPU/RAM.
    Includes progress tracking, caching for resumption, and a fail-fast mechanism.
    Gracefully handles the structural warnings thrown by Keras inside tf.GradientTape.
    """
    n_samples = X.shape[0]
    shap_batches = []
    total_batches = (n_samples + batch_size - 1) // batch_size

    if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*The structure of `inputs` doesn't match.*")

        for b_idx, i in enumerate(range(0, n_samples, batch_size)):
            cache_file = os.path.join(cache_dir, f"batch_{b_idx}.pickle") if cache_dir else None

            # 1. Resume mechanism: Check if this batch is already computed
            if cache_file and os.path.exists(cache_file):
                print(f"  -> Loading batch {b_idx + 1}/{total_batches} from cache...")
                with open(cache_file, 'rb') as f:
                    batch_vals = pickle.load(f)
            else:
                # 2. Compute mechanism: Process through the model
                print(f"  -> Computing batch {b_idx + 1}/{total_batches}...")
                X_batch = X[i: i + batch_size]
                batch_vals = explainer.shap_values(X_batch)

                # Fail-fast check ONLY on newly computed first batch
                if b_idx == 0:
                    if isinstance(batch_vals, list):
                        is_all_nan = all(np.isnan(c).all() for c in batch_vals)
                    else:
                        is_all_nan = np.isnan(batch_vals).all()

                    if is_all_nan:
                        raise RuntimeError("FAIL FAST: The first SHAP batch returned ONLY NaNs. Aborting early.")

                # Save newly computed batch to cache
                if cache_file:
                    with open(cache_file, 'wb') as f:
                        pickle.dump(batch_vals, f)

            shap_batches.append(batch_vals)

            gc.collect()
            tf.keras.backend.clear_session()

    if isinstance(shap_batches[0], list):
        num_classes = len(shap_batches[0])
        merged_list = []
        for c in range(num_classes):
            merged_class = np.concatenate([b[c] for b in shap_batches], axis=0)
            merged_list.append(merged_class)
        return merged_list
    else:
        return np.concatenate(shap_batches, axis=0)


def generate_and_save_fallback_shap(
        model: Any,
        X_train: np.ndarray,
        X_test: np.ndarray,
        num_classes: int,
        dataset_path: str,
        bg_samples: int = 50,
        batch_size: int = 32
) -> None:
    """
    Generates fallback SHAP explanations using GradientExplainer with batching and caching.
    Safely cleans up cache directories only upon full completion.
    """
    print(f"INFO: Initiating Gradient SHAP fallback for {os.path.basename(dataset_path)}...")

    N_tr, T, V = X_train.shape
    N_ts = X_test.shape[0]

    cache_dir_tr = os.path.join(dataset_path, '.cache_shap_tr')
    cache_dir_ts = os.path.join(dataset_path, '.cache_shap_ts')

    print(f"INFO: Clustering {N_tr} training samples into {bg_samples} background centroids...")
    X_train_2d = X_train.reshape(N_tr, T * V)
    n_clusters = min(bg_samples, N_tr)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(X_train_2d)
    background_3d = kmeans.cluster_centers_.reshape(n_clusters, T, V)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*The structure of `inputs` doesn't match.*")
        explainer = shap.GradientExplainer(model, background_3d)

    print(f"INFO: Processing SHAP values for TRAIN set...")
    shap_tr_raw = compute_shap_in_batches(explainer, X_train, batch_size=batch_size, cache_dir=cache_dir_tr)

    print(f"INFO: Processing SHAP values for TEST set...")
    shap_ts_raw = compute_shap_in_batches(explainer, X_test, batch_size=batch_size, cache_dir=cache_dir_ts)

    shap_tr_4d = format_explanations_to_4d_strict(shap_tr_raw, N_tr, num_classes, T, V)
    shap_ts_4d = format_explanations_to_4d_strict(shap_ts_raw, N_ts, num_classes, T, V)

    if np.isnan(shap_tr_4d).all() or np.isnan(shap_ts_4d).all():
        raise RuntimeError(f"FAIL FAST: Fallback Gradient SHAP returned ONLY NaNs for {dataset_path}.")

    if np.isnan(shap_ts_4d).any():
        print("WARN: Partial NaNs detected in fallback SHAP values. Downstream processing might be affected.")

    tr_path = os.path.join(dataset_path, 'svtr.pickle')
    ts_path = os.path.join(dataset_path, 'svts.pickle')

    print(f"INFO: Saving final artifacts to {tr_path} and {ts_path}...")
    with open(tr_path, 'wb') as f:
        pickle.dump(shap_tr_raw, f)

    with open(ts_path, 'wb') as f:
        pickle.dump(shap_ts_raw, f)

    # Safe cleanup ONLY after a successful write
    print("INFO: Cleaning up temporary cache directories...")
    if os.path.exists(cache_dir_tr):
        shutil.rmtree(cache_dir_tr)
    if os.path.exists(cache_dir_ts):
        shutil.rmtree(cache_dir_ts)

    print("INFO: Fallback generation complete and successfully saved.")

In [29]:
# test_dataset_path = next(p for p in verified_dataset_paths if "univariate" in p)  # "multivariate"
test_dataset_path = next(p for p in verified_dataset_paths if "EthanolConcentration" in p)
# test_dataset_path = next(p for p in verified_dataset_paths if "ArticularyWordRecognition" in p)
ds_name = os.path.basename(test_dataset_path)

print(f"--- Processing {ds_name} step-by-step ---")

# 1. Ładowanie danych Treningowych i Testowych
with open(os.path.join(test_dataset_path, 'trainX.pickle'), 'rb') as f:
    trainX = pickle.load(f)
with open(os.path.join(test_dataset_path, 'testX.pickle'), 'rb') as f:
    testX = pickle.load(f)
with open(os.path.join(test_dataset_path, 'testy.pickle'), 'rb') as f:
    testy = pickle.load(f)

print(f"testX shape: {testX.shape}")

# 2. Ładowanie modelu
input_dim = testX.shape[1:]
num_classes = testy.shape[1] if testy.ndim > 1 else len(np.unique(testy))
model = load_benchmark_model(test_dataset_path, input_shape=input_dim, num_classes=num_classes)

# 3. Ładowanie atrybucji SHAP (Trening i Test)
with open(os.path.join(test_dataset_path, 'svtr.pickle'), 'rb') as f:
    shap_tr_raw = pickle.load(f)
with open(os.path.join(test_dataset_path, 'svts.pickle'), 'rb') as f:
    shap_ts_raw = pickle.load(f)
print(f"shap_tr_raw: {shap_tr_raw.shape}, shap_ts_raw: {shap_ts_raw.shape}")

tr_all_nan = np.isnan(shap_tr_raw).all()
ts_all_nan = np.isnan(shap_ts_raw).all()
print(f"SHAP TR: {tr_all_nan}, TS: {ts_all_nan}")

shap_tr_4d, success_tr = format_explanations_to_4d(shap_tr_raw, trainX.shape, num_classes)
shap_ts_4d, success_ts = format_explanations_to_4d(shap_ts_raw, testX.shape, num_classes)

print(f"shap_tr_4d: {shap_tr_4d.shape}, shap_ts_4d: {shap_ts_4d.shape}")

--- Processing EthanolConcentration step-by-step ---
testX shape: (131, 1751, 3)
shap_tr_raw: (393, 1751, 3), shap_ts_raw: (131, 1751, 3)
SHAP TR: True, TS: True
WARN: Formatted explanation array contains ONLY NaN values.
WARN: Formatted explanation array contains ONLY NaN values.
shap_tr_4d: (393, 4, 1751, 3), shap_ts_4d: (131, 4, 1751, 3)


In [7]:
def verify_and_repair_shap_artifacts(verified_paths: List[str]) -> None:
    """
    Iterates over all verified datasets, checks the integrity of SHAP artifacts,
    and triggers the fallback generation if they are corrupted (e.g., only NaNs).
    Optimized to load the Keras model only when a fallback is strictly necessary.
    """
    print("Starting SHAP artifact verification and repair loop...\n")

    for dataset_path in verified_paths:
        ds_name = os.path.basename(dataset_path)
        print(f"--- Scaning dataset: {ds_name} ---")

        with open(os.path.join(dataset_path, 'trainX.pickle'), 'rb') as f:
            trainX = pickle.load(f)
        with open(os.path.join(dataset_path, 'testX.pickle'), 'rb') as f:
            testX = pickle.load(f)
        with open(os.path.join(dataset_path, 'testy.pickle'), 'rb') as f:
            testy = pickle.load(f)

        input_dim = testX.shape[1:]
        num_classes = testy.shape[1] if testy.ndim > 1 else len(np.unique(testy))

        shap_tr_path = os.path.join(dataset_path, 'svtr.pickle')
        shap_ts_path = os.path.join(dataset_path, 'svts.pickle')

        needs_fallback = False

        # Attempt to load and format existing SHAP artifacts
        try:
            with open(shap_tr_path, 'rb') as f:
                shap_tr_raw = pickle.load(f)
            with open(shap_ts_path, 'rb') as f:
                shap_ts_raw = pickle.load(f)

            _, tr_valid = format_explanations_to_4d(shap_tr_raw, trainX.shape, num_classes)
            _, ts_valid = format_explanations_to_4d(shap_ts_raw, testX.shape, num_classes)

            if not tr_valid or not ts_valid:
                needs_fallback = True

        except Exception as e:
            print(f"WARN: Failed to load or format SHAP for {ds_name} ({e}).")
            needs_fallback = True

        # Trigger fallback generator if artifacts are missing, corrupted, or full of NaNs
        if needs_fallback:
            print(f"WARN: Corrupted SHAP artifacts detected for {ds_name}. Initiating fallback generator.")

            # Load model lazily only when fallback is required
            try:
                model = load_benchmark_model(dataset_path, input_shape=input_dim, num_classes=num_classes)

                generate_and_save_fallback_shap(
                    model=model,
                    X_train=trainX,
                    X_test=testX,
                    num_classes=num_classes,
                    dataset_path=dataset_path,
                    batch_size=32
                )

                del model

            except Exception as fallback_error:
                print(f"ERROR: Fallback completely failed for {ds_name}: {fallback_error}")
        else:
            print(f"OK: SHAP artifacts for {ds_name} are healthy.")

        # Aggressive memory cleanup for arrays loaded in this iteration
        del trainX, testX, testy
        if 'shap_tr_raw' in locals(): del shap_tr_raw
        if 'shap_ts_raw' in locals(): del shap_ts_raw

        tf.keras.backend.clear_session()
        gc.collect()

    print("\nSHAP verification and repair loop completed.")


# --- Execution ---
verify_and_repair_shap_artifacts(verified_dataset_paths)

Starting SHAP artifact verification and repair loop...

--- Scaning dataset: Adiac ---
OK: SHAP artifacts for Adiac are healthy.
--- Scaning dataset: BME ---
OK: SHAP artifacts for BME are healthy.
--- Scaning dataset: Beef ---
OK: SHAP artifacts for Beef are healthy.
--- Scaning dataset: BeetleFly ---
OK: SHAP artifacts for BeetleFly are healthy.
--- Scaning dataset: BirdChicken ---
OK: SHAP artifacts for BirdChicken are healthy.
--- Scaning dataset: CBF ---
OK: SHAP artifacts for CBF are healthy.
--- Scaning dataset: Chinatown ---
OK: SHAP artifacts for Chinatown are healthy.
--- Scaning dataset: Coffee ---
OK: SHAP artifacts for Coffee are healthy.
--- Scaning dataset: Computers ---
OK: SHAP artifacts for Computers are healthy.
--- Scaning dataset: CricketX ---
OK: SHAP artifacts for CricketX are healthy.
--- Scaning dataset: CricketY ---
OK: SHAP artifacts for CricketY are healthy.
--- Scaning dataset: CricketZ ---
OK: SHAP artifacts for CricketZ are healthy.
--- Scaning dataset: C

2026-03-01 20:35:38.474505: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


  -> Computing batch 2/13...
  -> Computing batch 3/13...
  -> Computing batch 4/13...
  -> Computing batch 5/13...
  -> Computing batch 6/13...
  -> Computing batch 7/13...
  -> Computing batch 8/13...
  -> Computing batch 9/13...
  -> Computing batch 10/13...
  -> Computing batch 11/13...
  -> Computing batch 12/13...
  -> Computing batch 13/13...
INFO: Processing SHAP values for TEST set...
  -> Computing batch 1/5...
  -> Computing batch 2/5...
  -> Computing batch 3/5...
  -> Computing batch 4/5...
  -> Computing batch 5/5...
ERROR: Fallback completely failed for EthanolConcentration: Shape mismatch. Expected (393, 4, 1751, 3), got (393, 1751, 3, 4)
--- Scaning dataset: FaceDetection ---
OK: SHAP artifacts for FaceDetection are healthy.
--- Scaning dataset: FingerMovements ---
OK: SHAP artifacts for FingerMovements are healthy.
--- Scaning dataset: HandMovementDirection ---
OK: SHAP artifacts for HandMovementDirection are healthy.
--- Scaning dataset: Handwriting ---
OK: SHAP arti

## Fix EthanolConcentration

In [9]:
def merge_cached_batches(cache_dir: str) -> Any:
    """
    Loads and merges sequential SHAP batch files from a given cache directory.
    """
    batch_files = [f for f in os.listdir(cache_dir) if f.startswith('batch_') and f.endswith('.pickle')]
    batch_files.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))

    print(f"INFO: Merging {len(batch_files)} batches from {os.path.basename(cache_dir)}...")
    shap_batches = []

    for file in batch_files:
        with open(os.path.join(cache_dir, file), 'rb') as f:
            shap_batches.append(pickle.load(f))

    if isinstance(shap_batches[0], list):
        num_classes = len(shap_batches[0])
        merged_list = []
        for c in range(num_classes):
            merged_class = np.concatenate([b[c] for b in shap_batches], axis=0)
            merged_list.append(merged_class)
        return merged_list
    else:
        return np.concatenate(shap_batches, axis=0)

In [30]:
# --- STEP 1: Setup Paths and Load Dimensions ---
dataset_type = "multivariate"
dataset_name = "EthanolConcentration"
base_dir = "/home/jovyan/shared/explain-ts/ds"
dataset_path = os.path.join(base_dir, dataset_type, dataset_name)

print(f"--- Processing {dataset_name} ---")

# Load ground truth arrays to extract dimensions
with open(os.path.join(dataset_path, 'trainX.pickle'), 'rb') as f:
    trainX = pickle.load(f)
with open(os.path.join(dataset_path, 'testX.pickle'), 'rb') as f:
    testX = pickle.load(f)
with open(os.path.join(dataset_path, 'testy.pickle'), 'rb') as f:
    testy = pickle.load(f)

N_tr, T, V = trainX.shape
N_ts = testX.shape[0]
num_classes = testy.shape[1] if testy.ndim > 1 else len(np.unique(testy))

print(f"N_tr: {N_tr}, N_ts: {N_ts}, T: {T}, V: {V}, num_classes: {num_classes}")

with open(os.path.join(dataset_path, 'svtr.pickle'), 'rb') as f:
    svtr = pickle.load(f)
with open(os.path.join(dataset_path, 'svts.pickle'), 'rb') as f:
    svts = pickle.load(f)
print(f"svtr: {svtr.shape}, shap_ts_4d: {svts.shape}")

--- Processing EthanolConcentration ---
N_tr: 393, N_ts: 131, T: 1751, V: 3, num_classes: 4
svtr: (393, 1751, 3), shap_ts_4d: (131, 1751, 3)


In [31]:
# --- STEP 2: Merge Cached Batches ---
cache_dir_tr = os.path.join(dataset_path, '.cache_shap_tr')
cache_dir_ts = os.path.join(dataset_path, '.cache_shap_ts')

shap_tr_raw = merge_cached_batches(cache_dir_tr)
shap_ts_raw = merge_cached_batches(cache_dir_ts)

print(f"shap_tr_raw: {shap_tr_raw.shape}, shap_ts_raw: {shap_ts_raw.shape}")

# --- STEP 3: Format and Validate Dimensions ---
print("INFO: Formatting and validating array dimensions...")
shap_tr_4d = format_explanations_to_4d_strict(shap_tr_raw, N_tr, num_classes, T, V)
shap_ts_4d = format_explanations_to_4d_strict(shap_ts_raw, N_ts, num_classes, T, V)

print(f"shap_tr_4d: {shap_tr_4d.shape}, shap_ts_4d: {shap_ts_4d.shape}")

print("SUCCESS: Dimensions are correct.")

# --- STEP 3.5: NaN Verification ---
print("INFO: Checking for NaN values in recovered SHAP arrays...")

tr_all_nan = np.isnan(shap_tr_4d).all()
ts_all_nan = np.isnan(shap_ts_4d).all()

if tr_all_nan or ts_all_nan:
    raise ValueError("CRITICAL: Recovered SHAP values contain ONLY NaNs. The model's gradients might be broken.")

tr_any_nan = np.isnan(shap_tr_4d).any()
ts_any_nan = np.isnan(shap_ts_4d).any()

if tr_any_nan or ts_any_nan:
    print("WARN: Partial NaNs detected in the arrays. Downstream processing might be affected, but data is not entirely lost.")
else:
    print("SUCCESS: No NaNs detected. The SHAP artifacts are completely clean and healthy!")

INFO: Merging 13 batches from .cache_shap_tr...
INFO: Merging 5 batches from .cache_shap_ts...
shap_tr_raw: (393, 1751, 3, 4), shap_ts_raw: (131, 1751, 3, 4)
INFO: Formatting and validating array dimensions...
shap_tr_4d: (393, 4, 1751, 3), shap_ts_4d: (131, 4, 1751, 3)
SUCCESS: Dimensions are correct.
INFO: Checking for NaN values in recovered SHAP arrays...
SUCCESS: No NaNs detected. The SHAP artifacts are completely clean and healthy!


In [32]:
# --- STEP 4: Overwrite Corrupted Pickles ---
svtr_path = os.path.join(dataset_path, 'svtr.pickle')
svts_path = os.path.join(dataset_path, 'svts.pickle')

print("INFO: Saving recovered artifacts to disk...")
with open(svtr_path, 'wb') as f:
    pickle.dump(shap_tr_raw, f)
with open(svts_path, 'wb') as f:
    pickle.dump(shap_ts_raw, f)


# --- STEP 5: Create Zenodo-Compliant ZIP Archive ---
# Construct the specific Zenodo filename
zip_filename = f"{dataset_type}_{dataset_name}_shap_values.zip"
zip_filepath = os.path.join(dataset_path, zip_filename)

print(f"INFO: Creating Zenodo archive: {zip_filename}...")
with zipfile.ZipFile(zip_filepath, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add files explicitly without subdirectories (arcname defines internal path)
    zipf.write(svtr_path, arcname='svtr.pickle')
    zipf.write(svts_path, arcname='svts.pickle')

print(f"SUCCESS: Archive saved at {zip_filepath}")

INFO: Saving recovered artifacts to disk...
INFO: Creating Zenodo archive: multivariate_EthanolConcentration_shap_values.zip...
SUCCESS: Archive saved at /home/jovyan/shared/explain-ts/ds/multivariate/EthanolConcentration/multivariate_EthanolConcentration_shap_values.zip
